# Task 2: Baseline generation

**Settings:** Accelerator **GPU T4 x2**, Internet **on**. Run with **Save Version > Save & Run All (Commit)**.

The base model `Qwen/Qwen2.5-0.5B` answers the fixed AlpacaEval subset
(`eval_sets/alpaca_eval_300_seed42.json`: 300 of the 805 prompts, chosen with seed 42).
Greedy decoding runs twice in parallel, one variant per GPU: plain greedy, and greedy with a
modest repetition penalty of 1.1. Comparing the two decides the decoding setting used for the
SFT and DPO comparisons in Task 6. A 5-prompt sanity check runs first.
Everything is archived as `task2_baseline_outputs.zip`.

In [ ]:
REPO_URL = "https://github.com/mabdullahi7780/small-model-alignment-starter.git"
BRANCH = "main"
WORK_DIR = "/kaggle/working"
SCRATCH_DIR = "/tmp"
REPO_DIR = f"{SCRATCH_DIR}/small-model-alignment-starter"
OUT_DIR = f"{WORK_DIR}/outputs/baseline"

MODEL = "Qwen/Qwen2.5-0.5B"
GENERATOR = "qwen2.5-0.5b-base"
PROMPTS = "eval_sets/alpaca_eval_300_seed42.json"
BATCH_SIZE = 32
MAX_NEW_TOKENS = 512
# name -> (GPU, repetition penalty)
VARIANTS = {"greedy": (0, 1.0), "greedy_rp1.1": (1, 1.1)}
RUN_SETTINGS = {"model": MODEL, "generator": GENERATOR, "prompts": PROMPTS, "batch_size": BATCH_SIZE,
                "max_new_tokens": MAX_NEW_TOKENS, "variants": VARIANTS, "dtype": "float32"}

In [ ]:
import os
import subprocess
import sys


def run(cmd, cwd=None, env=None, check=True):
    print("$", " ".join(cmd), flush=True)
    return subprocess.run(cmd, cwd=cwd, env=env, check=check)


if not os.path.isdir(REPO_DIR):
    run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR])
run(["git", "log", "-1", "--format=commit %H%nauthor date %ad%nsubject %s"], cwd=REPO_DIR)

In [ ]:
# Kaggle's preinstalled torch is built for its GPU runtime; pin it so pip can never replace it.
TORCH_BEFORE = subprocess.run(
    [sys.executable, "-c", "import torch; print(torch.__version__)"],
    capture_output=True, text=True, check=True,
).stdout.strip()
print("Preinstalled torch:", TORCH_BEFORE)

constraint_file = f"{SCRATCH_DIR}/torch-constraint.txt"
with open(constraint_file, "w") as handle:
    handle.write(f"torch=={TORCH_BEFORE}\n")

run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt", "-c", constraint_file], cwd=REPO_DIR)
run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"], cwd=REPO_DIR)

In [ ]:
import json

COMMIT = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, capture_output=True, text=True, check=True).stdout.strip()
os.makedirs(f"{OUT_DIR}/logs", exist_ok=True)
with open(f"{OUT_DIR}/run_info.json", "w") as handle:
    json.dump({"commit": COMMIT, "torch": TORCH_BEFORE, **RUN_SETTINGS}, handle, indent=2)
print("commit", COMMIT)

In [ ]:
def run_on_gpus(jobs, tag):
    """Start one subprocess per job, each pinned to its own GPU, and wait for all of them.

    jobs: list of (name, gpu_index, argv). Each job's output goes to logs/<tag>_<name>.log.
    """
    env_base = {**os.environ, "PYTHONPATH": f"{REPO_DIR}/src:{REPO_DIR}"}
    running = []
    for name, gpu, argv in jobs:
        log_path = f"{OUT_DIR}/logs/{tag}_{name}.log"
        handle = open(log_path, "w")
        print(f"start {tag}/{name} on GPU {gpu}", flush=True)
        process = subprocess.Popen(
            [sys.executable, *argv], cwd=REPO_DIR, stdout=handle, stderr=subprocess.STDOUT,
            env={**env_base, "CUDA_VISIBLE_DEVICES": str(gpu)},
        )
        running.append((name, process, handle, log_path))
    failed = []
    for name, process, handle, log_path in running:
        code = process.wait()
        handle.close()
        print(f"\n===== {tag}/{name}: exit code {code} =====")
        print(open(log_path).read()[-8000:])
        if code:
            failed.append(name)
    if failed:
        raise RuntimeError(f"{tag}: failed jobs {failed}; see {OUT_DIR}/logs")

## Download the model once

Both jobs read the same local copy instead of downloading in parallel.

In [ ]:
run([sys.executable, "-c", f"from huggingface_hub import snapshot_download; print(snapshot_download({MODEL!r}))"])

## Sanity check: first 5 prompts per variant

Inspect these before trusting the full run: prompt format, stopping, length.

In [ ]:
def generation_jobs(tag, extra):
    return [
        (name, gpu, ["-m", "pa2.generate", "--model", MODEL, "--generator", GENERATOR, "--prompts", PROMPTS,
                     "--out", f"{OUT_DIR}/{tag}/{GENERATOR}_{name}.json", "--batch-size", str(BATCH_SIZE),
                     "--max-new-tokens", str(MAX_NEW_TOKENS), "--repetition-penalty", str(penalty), *extra])
        for name, (gpu, penalty) in VARIANTS.items()
    ]


run_on_gpus(generation_jobs("sanity", ["--limit", "5", "--show", "5"]), "sanity")

## Full run: 300 prompts per variant

In [ ]:
run_on_gpus(generation_jobs("alpaca_eval", ["--show", "3"]), "alpaca_eval")

## Compare the two decoding variants

In [ ]:
rows = {}
for name in VARIANTS:
    path = f"{OUT_DIR}/alpaca_eval/{GENERATOR}_{name}"
    meta = json.load(open(path + ".meta.json"))
    rows[name] = {**meta["stats"], "seconds": meta["seconds"]}
    records = json.load(open(path + ".json"))
    assert len(records) == 300 and all(r["generator"] == GENERATOR for r in records)

keys = list(next(iter(rows.values())))
print(f"{'metric':<22}" + "".join(f"{name:>16}" for name in rows))
for key in keys:
    print(f"{key:<22}" + "".join(f"{rows[name][key]:>16.3f}" for name in rows))
json.dump(rows, open(f"{OUT_DIR}/variant_comparison.json", "w"), indent=2)

outputs = {name: json.load(open(f"{OUT_DIR}/alpaca_eval/{GENERATOR}_{name}.json")) for name in VARIANTS}
for index in (0, 100, 200):
    print("\n" + "=" * 100)
    print("PROMPT:", outputs["greedy"][index]["instruction"][:300])
    for name in VARIANTS:
        record = outputs[name][index]
        print(f"\n--- {name} [{record['stop_reason']}, {record['new_tokens']} tokens]")
        print(record["output"][:700])

## Archive the outputs

In [ ]:
import shutil

archive = shutil.make_archive(f"{WORK_DIR}/task2_baseline_outputs", "zip", OUT_DIR)
print("Saved", archive)
for root, _, files in os.walk(OUT_DIR):
    for name in sorted(files):
        print(os.path.relpath(os.path.join(root, name), OUT_DIR))